💡 **Environment:** `clamp-analyses`  


# Description

Predicts drug-disease associations using the recount2 **module-based** CLAMP model.

Reads LINCS and S-PrediXcan projections from:
- `04_spredixcan_projection_recount2` (S-PrediXcan, LVs × traits)
- `05_lincs_projection_recount2` (LINCS, LVs × drugs)

For each of 49 tissues and 5 LV-count thresholds (all, 5, 10, 25, 50), scores:
$$\text{score} = -1 \times \mathbf{drug}^T \mathbf{disease}$$


# Modules loading


In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from pyprojroot import here

# Settings


In [ ]:
PREDICTION_METHOD = 'module_based_recount2'
MODEL_KEY = PREDICTION_METHOD.removeprefix('module_based_')

In [4]:
DATA_DIR = here('data/drug_disease_associations')
display(DATA_DIR)
assert DATA_DIR.exists()

NB_NAME = '09_prediction_module_based_recount2'
OUTPUT_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/' + NB_NAME)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs from upstream projection notebooks
LINCS_PROJ_FILE = here('output/03_model_biology/00_archs4/02_drug_disease_associations/05_lincs_projection_recount2') / 'lincs' / 'lincs-projection.pkl'
display(LINCS_PROJ_FILE)
assert LINCS_PROJ_FILE.exists()

SPREDIXCAN_PROJ_DIR = here('output/03_model_biology/00_archs4/02_drug_disease_associations/04_spredixcan_projection_recount2') / 'spredixcan'
display(SPREDIXCAN_PROJ_DIR)
assert SPREDIXCAN_PROJ_DIR.exists()

OUTPUT_PREDICTIONS_DIR = OUTPUT_DIR / 'lincs' / 'predictions'
OUTPUT_PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
display(OUTPUT_PREDICTIONS_DIR)


PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/data/drug_disease_associations')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/05_lincs_projection_recount2/lincs/lincs-projection.pkl')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/04_spredixcan_projection_recount2/spredixcan')

PosixPath('/home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions')

# Helper functions


In [5]:
import sys
sys.path.insert(0, str(here('libs')))
from drug_disease_utils import map_traits_to_doid, _zero_nontop_genes, predict_dotprod_neg

# Load PharmacotherapyDB gold standard


In [6]:
gold_standard = pd.read_pickle(DATA_DIR / 'gold_standard.pkl')
display(gold_standard.shape)
display(gold_standard.head())

doids_in_gold_standard = set(gold_standard['trait'])
print(f'Unique DOIDs in gold standard: {len(doids_in_gold_standard)}')

(998, 3)

,trait,drug,true_class
0,DOID:10652,DB00843,1
1,DOID:10652,DB00674,1
2,DOID:10652,DB01043,1
3,DOID:10652,DB00989,1
4,DOID:10652,DB00810,0


Unique DOIDs in gold standard: 87


# Load trait → DOID mapping files


In [7]:
ukb_efo = pd.read_csv(
    DATA_DIR / 'phenomexcan_traits_fullcode_to_efo.tsv',
    sep='\t',
    index_col='ukb_fullcode',
)
# PhenoPlier stores trait full codes with hyphens (e.g. "I70-Diagnoses_...") but
# our S-PrediXcan data uses underscores throughout (e.g. "I70_Diagnoses_...").
# Normalize the index so lookups work correctly.
ukb_efo.index = [idx.replace('-', '_', 1) for idx in ukb_efo.index]

efo_xrefs = pd.read_csv(DATA_DIR / 'term_id_xrefs.tsv.gz', sep='\t')
do_xrefs = pd.read_csv(DATA_DIR / 'xrefs-prop-slim.tsv', sep='\t')

# Load LINCS projection


In [8]:
lincs_projection = pd.read_pickle(LINCS_PROJ_FILE)
print(f'LINCS projection shape: {lincs_projection.shape}')
assert not lincs_projection.isna().any().any()
display(lincs_projection.head())


LINCS projection shape: (724, 1170)


perturbagen,DB00014,DB00091,DB00121,DB00130,DB00131,DB00132,DB00136,DB00140,DB00146,DB00150,...,DB08995,DB09002,DB09004,DB09009,DB09010,DB09015,DB09019,DB09020,DB09022,DB09023
LV1,0.003179,-0.215170,0.060520,-0.096349,0.024118,0.007310,0.086585,-0.004956,-0.042509,-0.010136,...,0.037589,0.079428,-0.024861,-0.025116,-0.070027,0.038477,0.045107,0.128886,0.054431,-0.007958
LV2,0.023899,-0.042962,0.009088,-0.033447,0.009802,-0.011031,0.112337,-0.000358,0.009663,0.009500,...,-0.048660,0.000381,-0.001468,-0.010170,0.013613,-0.029456,-0.059639,-0.012081,0.026457,-0.003287
LV3,-0.008056,0.065900,0.065524,0.023932,-0.040531,0.099507,-0.018339,0.016091,0.022351,-0.015659,...,-0.038714,-0.003951,0.021199,0.030703,-0.083161,-0.000687,0.059978,0.040192,0.042454,-0.000282
LV4,0.001010,-0.014094,0.023886,0.000340,0.009103,0.008644,-0.036792,0.017644,-0.024312,-0.001480,...,0.031384,0.026598,0.005721,0.015177,0.003279,0.001539,0.045908,0.111801,0.030689,0.000260
LV5,-0.012695,0.252748,-0.075861,-0.001037,-0.015189,-0.172504,-0.064784,-0.012453,0.005494,-0.009908,...,-0.004436,-0.021837,-0.034371,-0.043902,0.049067,0.039357,-0.073870,0.105535,-0.049924,0.010845


# Load S-PrediXcan projected files


In [9]:
spredixcan_file_list = sorted(
    f for f in SPREDIXCAN_PROJ_DIR.glob(f'spredixcan-*-projection-{MODEL_KEY}.pkl')
)
display(len(spredixcan_file_list))
assert len(spredixcan_file_list) == 49

display(pd.read_pickle(spredixcan_file_list[0]).head())


49

,100001_raw_Food_weight,100002_raw_Energy,100003_raw_Protein,100004_raw_Fat,100005_raw_Carbohydrate,100006_raw_Saturated_fat,100007_raw_Polyunsaturated_fat,100008_raw_Total_sugars,100009_raw_Englyst_dietary_fibre,100010_Portion_size,...,Z50_Diagnoses_main_ICD10_Z50_Care_involving_use_of_rehabilitation_procedures,Z51_Diagnoses_main_ICD10_Z51_Other_medical_care,Z52_Diagnoses_main_ICD10_Z52_Donors_of_organs_and_tissues,Z53_Diagnoses_main_ICD10_Z53_Persons_encountering_health_services_for_specifie_procedures_not_carried_out,Z71_Diagnoses_main_ICD10_Z71_Persons_encountering_health_services_for_other_counselling_and_medical_advice_not_elsewhere_classified,Z76_Diagnoses_main_ICD10_Z76_Persons_encountering_health_services_in_other_circumstances,Z80_Diagnoses_main_ICD10_Z80_Family_history_of_malignant_neoplasm,Z85_Diagnoses_main_ICD10_Z85_Personal_history_of_malignant_neoplasm,Z87_Diagnoses_main_ICD10_Z87_Personal_history_of_other_diseases_and_conditions,pgc_scz2
LV1,0.039794,0.046069,0.009841,0.018781,0.047456,-0.005252,0.040445,0.024775,0.057574,0.020951,...,-0.004098,0.000362,0.027889,-0.045855,0.027800,-0.004065,-0.016176,0.052489,0.001312,-0.019163
LV2,-0.011789,0.038919,0.035305,0.048799,0.026463,0.046999,0.045454,-0.011395,0.006221,-0.020663,...,-0.014566,-0.002730,0.059988,0.028960,-0.056943,0.010547,0.009364,-0.007506,-0.000995,0.007736
LV3,0.007893,0.020056,0.006681,0.009169,0.012746,0.034276,-0.010672,0.019427,-0.015180,-0.039851,...,0.035515,0.052538,-0.009259,0.007969,-0.048234,0.009265,-0.051837,-0.042863,-0.016994,0.060708
LV4,0.013292,-0.022417,-0.014454,-0.027249,0.002355,-0.012414,-0.045302,0.002732,0.029403,-0.002895,...,0.009730,0.026667,0.026985,-0.005841,-0.033131,-0.015251,-0.049521,0.023363,0.001293,0.007640
LV5,0.039073,0.041021,0.057917,0.026472,0.030079,0.038741,0.013093,0.042051,0.009709,0.008790,...,0.028771,0.041647,-0.010597,0.004578,0.011447,-0.071790,0.017534,-0.021969,-0.003110,0.029381


# Predict drug-disease associations


In [10]:
# LV thresholds: None = all LVs (same as PhenoPlier module-based thresholds)
N_TOP_LVS_LIST = [None, 5, 10, 25, 50]

for spredixcan_file in spredixcan_file_list:
    print(spredixcan_file.name)

    tissue_proj = pd.read_pickle(spredixcan_file)
    print(f'  shape: {tissue_proj.shape}')
    assert tissue_proj.index.equals(lincs_projection.index)

    for ntc in N_TOP_LVS_LIST:
        predict_dotprod_neg(
            lincs_projection,
            spredixcan_file,
            tissue_proj,
            OUTPUT_PREDICTIONS_DIR,
            PREDICTION_METHOD,
            doids_in_gold_standard,
            ukb_efo,
            efo_xrefs,
            do_xrefs,
            n_top_conditions=ntc,
            use_abs=True,
        )

    print('')


spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-recount2.pkl
  shape: (724, 4091)
  predicting all_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-recount2-all_genes-prediction_scores.h5
  predicting top_5_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Adipose_Subcutaneous-projection-recount2-top_5_genes-prediction_scores.h5
  predicting top_10_genes...


...

    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-recount2-top_10_genes-prediction_scores.h5
  predicting top_25_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-recount2-top_25_genes-prediction_scores.h5
  predicting top_50_genes...


    shape: (1170, 4091)


    saved to: /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/09_prediction_module_based_recount2/lincs/predictions/dotprod_neg/spredixcan-mashr-zscores-Whole_Blood-projection-recount2-top_50_genes-prediction_scores.h5

